In [ ]:
!pip install transformers --quiet


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

# --------------------------------------------------
# 1️⃣ Load FULL training data
# --------------------------------------------------
train_df = pd.read_csv("tamil_text_cleaned.csv")

train_texts = train_df["clean_text"].astype(str).tolist()
train_labels = train_df["label"].astype(int).tolist()

print("Training samples:", len(train_texts))
print("Label distribution:")
print(train_df["label"].value_counts())

# --------------------------------------------------
# 2️⃣ Tokenizer
# --------------------------------------------------
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --------------------------------------------------
# 3️⃣ Dataset class
# --------------------------------------------------
class DepressionDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=128
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = DepressionDataset(train_texts, train_labels)

# --------------------------------------------------
# 4️⃣ Model (FULL fine-tuning)
# --------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# --------------------------------------------------
# 5️⃣ Training arguments (FINAL TRAINING)
# --------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

# --------------------------------------------------
# 6️⃣ Trainer (NO validation now)
# --------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

# --------------------------------------------------
# 7️⃣ Train on FULL data
# --------------------------------------------------
trainer.train()

# --------------------------------------------------
# 8️⃣ Load TEST text
# --------------------------------------------------
test_df = pd.read_csv("tamil_test_text.csv")

test_texts = test_df["clean_text"].astype(str).tolist()

test_dataset = DepressionDataset(test_texts)

# --------------------------------------------------
# 9️⃣ Predict on TEST data
# --------------------------------------------------
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(axis=1)

# --------------------------------------------------
# 🔟 Convert numeric → string labels
# --------------------------------------------------
label_map = {
    1: "Depressed",
    0: "Non-depressed"
}

test_df["label"] = [label_map[p] for p in pred_labels]

# --------------------------------------------------
# 1️⃣1️⃣ Create submission file
# --------------------------------------------------
submission = test_df[["file_name", "label"]]
submission.to_csv("submission_roberta.csv", index=False, encoding="utf-8")

print("✅ submission_roberta.csv created successfully!")
submission.head()


Training samples: 1364
Label distribution:
label
0    911
1    453
Name: count, dtype: int64


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.419167
100,0.083735
150,0.197387
200,0.036050
250,0.104920
300,0.016842
350,0.108345
400,0.085346
450,0.021795
500,0.055067


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ submission_roberta.csv created successfully!


,file_name,label
0,t1.wav,Depressed
1,t10.wav,Non-depressed
2,t100.wav,Depressed
3,t101.wav,Depressed
4,t102.wav,Depressed


In [ ]:
!unzip -q Test-set-tamil.zip -d Test-set-tamil


In [ ]:
import whisper
import os
import pandas as pd
import re
model = whisper.load_model("medium")


100%|█████████████████████████████████████| 1.42G/1.42G [00:18<00:00, 84.0MiB/s]


In [ ]:
def clean_tamil_text(text):
    text = str(text)

    # Remove quotes
    text = text.replace('"', '')
    text = text.replace('“', '').replace('”', '')

    # Remove trailing punctuation
    text = re.sub(r'[.,!?]+$', '', text)

    # Keep Tamil Unicode + space
    text = re.sub(r'[^\u0B80-\u0BFF\s]', ' ', text)

    # Normalize spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


In [ ]:
test_audio_dir = "Test-set-tamil/Test-set-tamil"   # change if needed

rows = []

audio_files = sorted([
    f for f in os.listdir(test_audio_dir)
    if f.endswith(".wav")
])

print("Total test audios:", len(audio_files))

for idx, file in enumerate(audio_files, 1):
    audio_path = os.path.join(test_audio_dir, file)

    print(f"[{idx}/{len(audio_files)}] Transcribing {file}")

    result = model.transcribe(
        audio_path,
        language="ta",
        fp16=True   # GPU acceleration
    )

    raw_text = result["text"]
    clean_text = clean_tamil_text(raw_text)

    rows.append({
        "file_name": file,
        "clean_text": clean_text
    })


Total test audios: 160
[1/160] Transcribing t1.wav
[2/160] Transcribing t10.wav
[3/160] Transcribing t100.wav
[4/160] Transcribing t101.wav
[5/160] Transcribing t102.wav
[6/160] Transcribing t103.wav
[7/160] Transcribing t104.wav
[8/160] Transcribing t105.wav
[9/160] Transcribing t106.wav
[10/160] Transcribing t107.wav
[11/160] Transcribing t108.wav
[12/160] Transcribing t109.wav
[13/160] Transcribing t11.wav
[14/160] Transcribing t110.wav
[15/160] Transcribing t111.wav
[16/160] Transcribing t112.wav
[17/160] Transcribing t113.wav
[18/160] Transcribing t114.wav
[19/160] Transcribing t115.wav
[20/160] Transcribing t116.wav
[21/160] Transcribing t117.wav
[22/160] Transcribing t118.wav
[23/160] Transcribing t119.wav
[24/160] Transcribing t12.wav
[25/160] Transcribing t120.wav
[26/160] Transcribing t121.wav
[27/160] Transcribing t122.wav
[28/160] Transcribing t123.wav
[29/160] Transcribing t124.wav
[30/160] Transcribing t125.wav
[31/160] Transcribing t126.wav
[32/160] Transcribing t127.wav

In [ ]:
test_df = pd.DataFrame(rows)

test_df.to_csv(
    "tamil_test_text.csv",
    index=False,
    encoding="utf-8"
)

print("✅ tamil_test_text.csv created successfully!")
test_df.head()


✅ tamil_test_text.csv created successfully!


,file_name,clean_text
0,t1.wav,நா பேசுவது கூட தகதி இல்லையா என் கஷ்டத்தைப் புர...
1,t10.wav,இரவு பகல் வருவதைப் போல இன்பும் முன் துண்பும் ம...
2,t100.wav,அவங்கள் எல்லாம் என்னைப்பற்றி என்ன நினைக்கிறீர்...
3,t101.wav,வாங்கு எல்லா என்ன பட்டி என்ன நினைக்கிறீர்கள் க...
4,t102.wav,நான் எங்கே இருக்கும் என்று தெரியலை யாரும் இங்க...
